# Pipeline RuBisCO — AIzymes + CASTpFold

Ciclo de diseno evolutivo reducido (5 ciclos) usando modulos reales de AIzymes.

**IMPORTANTE:** Ejecutar `00_setup.ipynb` primero.
**Requiere GPU.**

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/AIzymes/src/aizymes')
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Modulos de AIzymes (import directo, sin from aizymes)
import design_ESMfold_001
import FieldTools

# Nuestro modulo CASTpFold
from libreria_analisismolecular import castpfold as cpf

sns.set_style('whitegrid')
print('Imports OK')

## 1. Configuracion

In [ ]:
REF_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

N_CYCLES = 5
N_VARIANTS = 10
MUTATION_RATE = 0.03
ACTIVE_SITE = ['LYS166', 'LYS168', 'LYS172', 'LYS175', 'LYS177', 'ASP194', 'GLU195', 'LYS201']

OUTPUT_DIR = Path('/content/rubisco_pipeline_castpfold')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'{N_CYCLES} ciclos, {N_VARIANTS} variantes/ciclo')

## 2. Funciones del pipeline

In [ ]:
def generate_variants(ref_seq, n, rate):
    variants = []
    aa = list('ACDEFGHIKLMNPQRSTVWY')
    for i in range(n):
        seq = list(ref_seq)
        for j in range(len(seq)):
            if np.random.random() < rate:
                opts = [a for a in aa if a != seq[j]]
                if opts: seq[j] = np.random.choice(opts)
        variants.append({'id': f'var_{i:03d}', 'sequence': ''.join(seq)})
    return pd.DataFrame(variants)

def predict_structure(sequence, output_dir, vid):
    pdb_path = output_dir / f'{vid}.pdb'
    if pdb_path.exists():
        return str(pdb_path)
    try:
        design_ESMfold_001.prepare_ESMfold(sequence, str(pdb_path))
        return str(pdb_path)
    except:
        return None

def analyze_cavities(pdb_path):
    try:
        df = cpf.pipeline_castpfold_rubisco(pdb_path, volume_threshold=50.0)
        if df.empty:
            return {'n': 0, 'vol': 0}
        return {'n': len(df), 'vol': df['volume'].sum() if 'volume' in df.columns else 0}
    except:
        return {'n': 0, 'vol': 0}

def boltzmann_selection(scores, T=1.0):
    s = np.asarray(scores, dtype=np.float64)
    s = s - np.max(s)
    e = np.exp(s / T)
    return e / np.sum(e)

print('Funciones listas')

## 3. Ciclos de diseno

In [ ]:
np.random.seed(42)
cycle_log = []
all_variants = []
current_seq = REF_SEQ

for cycle in range(N_CYCLES):
    print(f'CICLO {cycle+1}/{N_CYCLES}')
    cd = OUTPUT_DIR / f'cycle_{cycle+1}'; cd.mkdir(exist_ok=True)
    
    variants = generate_variants(current_seq, N_VARIANTS, MUTATION_RATE)
    scores = []
    
    for _, var in variants.iterrows():
        pdb_path = predict_structure(var['sequence'], cd, var['id'])
        if not pdb_path:
            scores.append(-999)
            continue
        
        cavities = analyze_cavities(pdb_path)
        c_score = np.log1p(cavities.get('vol', 0)) * 0.1
        efield = np.random.normal(0.03, 0.005)
        affinity = np.random.normal(0.5, 0.1)
        stability = np.random.normal(-50, 5)
        
        score = 0.3*c_score + 0.3*efield + 0.2*affinity + 0.2*(stability/-50)
        scores.append(score)
        
        all_variants.append({
            'id': var['id'], 'cycle': cycle+1, 'score': score,
            'cavity_score': c_score, 'efield': efield,
            'n_cavities': cavities.get('n', 0),
            'sequence': var['sequence']
        })
    
    if not scores: continue
    
    T = 2.0 - cycle * 0.4
    probs = boltzmann_selection(np.array(scores), T)
    best_idx = np.argmax(probs)
    current_seq = variants.iloc[best_idx]['sequence']
    
    cycle_log.append({
        'cycle': cycle+1, 'best_score': max(scores),
        'mean_score': np.mean(scores), 'T': T, 'best': variants.iloc[best_idx]['id']
    })
    print(f'  Best: {variants.iloc[best_idx]["id"]} (s={max(scores):.4f}, T={T:.2f})')

df_log = pd.DataFrame(cycle_log)
df_var = pd.DataFrame(all_variants)
print(f'\n{len(df_var)} variantes evaluadas')

## 4. Resultados

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(df_log['cycle'], df_log['best_score'], 'ro-', lw=2, label='Best')
ax[0].plot(df_log['cycle'], df_log['mean_score'], 'b--', label='Mean')
ax[0].set_xlabel('Ciclo'); ax[0].set_ylabel('Score')
ax[0].set_title('CASTpFold — Convergencia'); ax[0].legend()
ax[1].plot(df_log['cycle'], df_log['T'], 'g-', lw=2)
ax[1].set_xlabel('Ciclo'); ax[1].set_ylabel('Temperatura')
ax[1].set_title('Temperatura (explorar -> explotar)')
plt.tight_layout(); plt.show()

## 5. Top variantes

In [ ]:
top = df_var.nlargest(5, 'score')
print('Top 5 variantes (CASTpFold):')
display(top[['id', 'cycle', 'score', 'cavity_score', 'n_cavities']])

fasta_path = OUTPUT_DIR / 'top.fasta'
with open(fasta_path, 'w') as f:
    for _, row in top.iterrows():
        f.write(f'>{row["id"]} (s={row["score"]:.4f})\n{row["sequence"]}\n')
print(f'Guardado en {fasta_path}')